# EDA 03 — FlavorDB and Molecular Similarity

This notebook consolidates the descriptive FlavorDB EDA and the molecular audit/similarity workflow.

**Inputs** (read-only):
- `dataset/processed/FlavorDB_entities.csv`
- `dataset/processed/FlavorDB_molecules.csv`
- `dataset/processed/FlavorDB_entity_molecules.csv`

**Outputs:** all CSV reports, text summaries, and plots are written to `EDA/results/flavour/`.

The primary molecular feature is **IDF-weighted Jaccard similarity**. It measures similarity between recorded FlavorDB molecular profiles; it is not, by itself, proof that two ingredients are valid substitutes.

Run the notebook from any working directory. The project root is located by searching upward from the notebook location for the `dataset/processed` folder.


## 1. Setup and project paths

In [1]:
from pathlib import Path
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "dataset" / "processed").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not locate project root. Expected a 'dataset/processed' directory "
        "above this notebook."
    )

try:
    NOTEBOOK_DIR = Path.cwd().resolve()
except NameError:
    NOTEBOOK_DIR = Path.cwd().resolve()

PROJECT_ROOT = find_project_root(NOTEBOOK_DIR)
DATA_DIR = PROJECT_ROOT / "dataset" / "processed"
OUTPUT_DIR = PROJECT_ROOT / "EDA" / "results" / "flavour"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root : {PROJECT_ROOT}")
print(f"Input folder : {DATA_DIR}")
print(f"Output folder: {OUTPUT_DIR}")


Project root : /Users/harshgupta/Coding/BTP_2
Input folder : /Users/harshgupta/Coding/BTP_2/dataset/processed
Output folder: /Users/harshgupta/Coding/BTP_2/EDA/results/flavour


## 2. Descriptive FlavorDB EDA\nEntity structure, categories, link integrity, molecule descriptors, distributions, and descriptive CSV outputs.

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

BASE_DIR = PROJECT_ROOT
DATA = BASE_DIR / "dataset" / "processed"
OUT = OUTPUT_DIR
OUT.mkdir(parents=True, exist_ok=True)

entities = pd.read_csv(DATA / "FlavorDB_entities.csv")
molecules = pd.read_csv(DATA / "FlavorDB_molecules.csv")
links = pd.read_csv(DATA / "FlavorDB_entity_molecules.csv")

print("=" * 60)
print("FLAVORDB EDA")
print("=" * 60)

print("\nEntities")
print(f"Rows: {len(entities)}")
print(f"Unique entity IDs: {entities['entity_id'].nunique()}")
print(f"Duplicate rows: {entities.duplicated().sum()}")
print(f"Duplicate entity IDs: {entities['entity_id'].duplicated().sum()}")

print("\nEntity categories:")
print(entities["category_readable"].value_counts())

print("\nEntity-Molecule links")
print(f"Rows: {len(links)}")
print(f"Unique entity-molecule pairs: {links.drop_duplicates(['entity_id', 'pubchem_id']).shape[0]}")
print(f"Duplicate pairs: {links.duplicated(['entity_id', 'pubchem_id']).sum()}")

print("\nMolecules")
print(f"Rows: {len(molecules)}")
print(f"Unique PubChem IDs: {molecules['pubchem_id'].nunique()}")
print(f"Duplicate PubChem IDs: {molecules['pubchem_id'].duplicated().sum()}")

molecules_per_entity = links.groupby("entity_id")["pubchem_id"].nunique()

print("\nMolecules per entity:")
print(molecules_per_entity.describe())

numeric = [
    "molecular_weight",
    "xlogp",
    "heavy_atom_count"
]

print("\nMolecular descriptor statistics:")
print(molecules[numeric].describe().T)

print("\nMissing values:")
print(molecules[numeric].isna().sum())

print("\nSkewness:")
print(molecules[numeric].skew())

print("\nMolecular descriptor percentiles:")
print(molecules[numeric].quantile([0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]))

for col in numeric:
    plt.figure(figsize=(8, 5))
    plt.hist(molecules[col].dropna(), bins=50)
    plt.xlabel(col)
    plt.ylabel("Frequency")
    plt.title(f"Distribution of {col}")
    plt.tight_layout()
    plt.savefig(OUT / f"{col}_distribution.png", dpi=150)
    plt.close()

    plt.figure(figsize=(8, 4))
    plt.boxplot(molecules[col].dropna(), vert=False)
    plt.xlabel(col)
    plt.title(f"Boxplot of {col}")
    plt.tight_layout()
    plt.savefig(OUT / f"{col}_boxplot.png", dpi=150)
    plt.close()

molecules_per_entity.describe().to_csv(
    OUT / "molecules_per_entity_statistics.csv"
)

molecules[numeric].describe().T.assign(
    skewness=molecules[numeric].skew()
).to_csv(
    OUT / "molecular_statistics.csv"
)

print(f"\nResults saved to: {OUT}")

FLAVORDB EDA

Entities
Rows: 936
Unique entity IDs: 936
Duplicate rows: 0
Duplicate entity IDs: 0

Entity categories:
category_readable
Fish                    120
Fruit                    81
Dish                     77
Herb                     51
Dairy                    48
Meat                     47
Plant                    45
Beverage Alcoholic       44
Essential Oil            42
Berry                    42
Seafood                  35
Bakery                   32
Spice                    25
Additive                 25
Vegetable                24
Cereal                   23
Legume                   23
Nut                      21
Plant Derivative         20
Cabbage                  16
Beverage                 13
Fungus                   11
Fruit Citrus             10
Vegetable Tuber          10
Flower                    9
Vegetable Root            8
Gourd                     8
Beverage Caffeinated      7
Maize                     6
Seed                      5
Vegetable Fruit         

## 3. Molecular profile audit

This section retains the calculations and review logic from `01_molecule_audit.py`:
- structural validation of entities, links, and molecule records
- entity-to-molecule profiles
- molecule frequency and smoothed IDF
- raw and IDF-weighted pairwise overlap
- identical-profile investigation
- explicit sanity-pair checks
- review-only ingredient-name matching

Low-evidence profiles and identical profiles are retained and flagged; they are not silently removed or treated as valid substitutions.


In [3]:
"""
01_molecule_audit.py

Final FlavorDB molecular-profile audit and similarity preparation.

Purpose
-------
This is the final audit step before using FlavorDB molecular similarity
inside the BTP substitution framework.

It:
1. validates FlavorDB entity/link/molecule consistency
2. builds entity -> molecule profiles
3. computes molecule frequency and smoothed IDF
4. computes raw and IDF-weighted molecular overlap
5. uses IDF-weighted Jaccard as the primary molecular similarity
6. investigates identical molecular profiles with categories and molecule metadata
7. generates candidate name matches for unresolved sanity-check names
8. saves a clean pairwise similarity table for downstream modeling

Important
---------
The molecular similarity score is NOT a complete substitution score.
It should later be combined with GraphSAGE/contextual compatibility,
carbon footprint, nutrition preservation, and hard constraints.

Run from GastroGraph-main:
    python src/01_molecule_audit.py
"""

from pathlib import Path
from collections import Counter, defaultdict
from difflib import SequenceMatcher
import hashlib
import math
import re

import numpy as np
import pandas as pd


# ================================================================
# 1. PATHS
# ================================================================

DATASET_ROOT = PROJECT_ROOT / "dataset"
PROCESSED_DIR = DATASET_ROOT / "processed"
CUSTOM_DATA_DIR = OUTPUT_DIR

ENTITIES_FILE = PROCESSED_DIR / "FlavorDB_entities.csv"
ENTITY_MOLECULES_FILE = PROCESSED_DIR / "FlavorDB_entity_molecules.csv"
MOLECULES_FILE = PROCESSED_DIR / "FlavorDB_molecules.csv"

ENTITY_STATS_FILE = OUTPUT_DIR / "molecule_audit_entity_stats.csv"
MOLECULE_FREQ_FILE = OUTPUT_DIR / "molecule_audit_molecule_frequency.csv"
PAIR_EXAMPLES_FILE = OUTPUT_DIR / "molecule_audit_pair_examples.csv"
PAIRWISE_FILE = OUTPUT_DIR / "molecule_audit_pairwise_overlap.csv"
IDENTICAL_PROFILES_FILE = OUTPUT_DIR / "molecule_audit_identical_profiles.csv"
IDENTICAL_ENTITIES_FILE = OUTPUT_DIR / "molecule_audit_identical_profile_entities.csv"
IDENTICAL_MOLECULES_FILE = OUTPUT_DIR / "molecule_audit_identical_profile_molecules.csv"
NAME_MATCH_FILE = OUTPUT_DIR / "molecule_audit_name_matching.csv"
NAME_CANDIDATES_FILE = OUTPUT_DIR / "molecule_audit_name_candidates.csv"
SUMMARY_FILE = OUTPUT_DIR / "molecule_audit_summary.txt"


# ================================================================
# 2. SETTINGS
# ================================================================

EXAMPLE_PAIRS = [
    ("beef", "beef stock"),
    ("chicken", "chicken breast"),
    ("tomato", "tomato paste"),
    ("olive oil", "extra virgin olive oil"),
    ("black pepper", "pepper"),
    ("onion", "yellow onion"),
    ("beef", "bay leaf"),
    ("beef", "wine vinegar"),
    ("chicken", "dill seed"),
    ("lobster", "tofu"),
    ("fish", "broccoli"),
]

# Profiles below this size are retained, but flagged as low evidence.
MIN_PROFILE_SIZE = 3

# Number of unresolved-name candidates to save.
TOP_NAME_CANDIDATES = 5

# A candidate must have at least this combined name score to be shown.
# Candidates are NEVER silently accepted as mappings.
MIN_CANDIDATE_SCORE = 0.35


# ================================================================
# 3. HELPERS
# ================================================================

def section(title):
    print("\n" + "=" * 72)
    print(title)
    print("=" * 72)


def clean_text(value):
    if pd.isna(value):
        return ""
    return str(value).strip()


def normalize_name(value):
    text = clean_text(value).lower()
    text = text.replace("&", " and ")
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def token_set(value):
    return set(normalize_name(value).split())


def jaccard(set_a, set_b):
    union = set_a | set_b
    return len(set_a & set_b) / len(union) if union else 0.0


def overlap_coefficient(set_a, set_b):
    minimum = min(len(set_a), len(set_b))
    return len(set_a & set_b) / minimum if minimum else 0.0


def symmetric_coverage(set_a, set_b):
    if not set_a or not set_b:
        return 0.0
    shared = len(set_a & set_b)
    return 0.5 * (shared / len(set_a) + shared / len(set_b))


def smoothed_idf(entity_count, molecule_frequency):
    if entity_count <= 0:
        return 0.0
    return math.log((entity_count + 1) / (molecule_frequency + 1)) + 1.0


def weighted_sum(molecules, weights):
    return sum(weights.get(m, 0.0) for m in molecules)


def profile_signature(molecule_set):
    raw = ",".join(str(x) for x in sorted(molecule_set))
    return hashlib.sha1(raw.encode("utf-8")).hexdigest()


def safe_join(values):
    return "|".join(str(x) for x in values)


def first_alias(aliases):
    return sorted(aliases)[0] if aliases else ""


def similarity_score(query, candidate):
    """
    Transparent name-candidate score.

    Combines:
    - exact normalized match
    - substring containment
    - token Jaccard
    - SequenceMatcher ratio

    This is ONLY for candidate generation.
    It is not used to change FlavorDB mappings automatically.
    """
    q = normalize_name(query)
    c = normalize_name(candidate)

    if not q or not c:
        return 0.0

    if q == c:
        return 1.0

    q_tokens = set(q.split())
    c_tokens = set(c.split())

    token_j = (
        len(q_tokens & c_tokens) / len(q_tokens | c_tokens)
        if q_tokens | c_tokens
        else 0.0
    )

    substring = 1.0 if (q in c or c in q) else 0.0
    sequence = SequenceMatcher(None, q, c).ratio()

    return (
        0.45 * token_j
        + 0.25 * substring
        + 0.30 * sequence
    )


# ================================================================
# 4. LOAD DATA
# ================================================================

def load_data():
    section("LOADING FLAVORDB DATA")

    required = [
        ENTITIES_FILE,
        ENTITY_MOLECULES_FILE,
        MOLECULES_FILE,
    ]

    for path in required:
        if not path.exists():
            raise FileNotFoundError(
                f"\nRequired file not found:\n{path}\n\n"
                "Check that the FlavorDB files are in dataset/processed/."
            )

    entities = pd.read_csv(ENTITIES_FILE)
    entity_molecules = pd.read_csv(ENTITY_MOLECULES_FILE)
    molecules = pd.read_csv(MOLECULES_FILE)

    print(f"FlavorDB entities     : {len(entities):,}")
    print(f"Entity-molecule links : {len(entity_molecules):,}")
    print(f"FlavorDB molecules    : {len(molecules):,}")

    return entities, entity_molecules, molecules


# ================================================================
# 5. BASIC DATA AUDIT
# ================================================================

def audit_basic_structure(entities, entity_molecules, molecules, summary):
    section("1. BASIC DATA INTEGRITY")

    required_entity_cols = {"entity_id"}
    required_link_cols = {"entity_id", "pubchem_id"}
    required_molecule_cols = {"pubchem_id"}

    if not required_entity_cols.issubset(entities.columns):
        raise ValueError("FlavorDB_entities.csv is missing entity_id.")

    if not required_link_cols.issubset(entity_molecules.columns):
        raise ValueError(
            "FlavorDB_entity_molecules.csv must contain entity_id and pubchem_id."
        )

    if not required_molecule_cols.issubset(molecules.columns):
        raise ValueError("FlavorDB_molecules.csv is missing pubchem_id.")

    duplicate_entity_ids = int(entities["entity_id"].duplicated().sum())
    duplicate_links = int(
        entity_molecules.duplicated(
            subset=["entity_id", "pubchem_id"]
        ).sum()
    )
    duplicate_molecule_ids = int(molecules["pubchem_id"].duplicated().sum())

    link_ids = set(
        pd.to_numeric(
            entity_molecules["pubchem_id"],
            errors="coerce",
        ).dropna().astype(int)
    )

    molecule_table_ids = set(
        pd.to_numeric(
            molecules["pubchem_id"],
            errors="coerce",
        ).dropna().astype(int)
    )

    missing_ids = link_ids - molecule_table_ids
    unused_ids = molecule_table_ids - link_ids

    print(f"Duplicate entity_id rows          : {duplicate_entity_ids:,}")
    print(f"Duplicate entity-molecule links   : {duplicate_links:,}")
    print(f"Duplicate molecule-table IDs      : {duplicate_molecule_ids:,}")
    print(f"Referenced molecule IDs           : {len(link_ids):,}")
    print(f"Molecule-table IDs                : {len(molecule_table_ids):,}")
    print(f"Referenced IDs missing in table   : {len(missing_ids):,}")
    print(f"Unused molecule-table IDs         : {len(unused_ids):,}")

    summary.extend([
        "BASIC DATA INTEGRITY",
        "-" * 72,
        f"Entities: {len(entities):,}",
        f"Entity-molecule links: {len(entity_molecules):,}",
        f"Molecule rows: {len(molecules):,}",
        f"Duplicate entity_id rows: {duplicate_entity_ids:,}",
        f"Duplicate entity-molecule links: {duplicate_links:,}",
        f"Duplicate molecule-table IDs: {duplicate_molecule_ids:,}",
        f"Referenced molecule IDs: {len(link_ids):,}",
        f"Molecule-table IDs: {len(molecule_table_ids):,}",
        f"Referenced IDs missing in molecule table: {len(missing_ids):,}",
        f"Unused molecule-table IDs: {len(unused_ids):,}",
        "",
    ])

    return missing_ids


# ================================================================
# 6. BUILD PROFILES + ALIASES
# ================================================================

def build_profiles(entities, entity_molecules):
    section("2. BUILDING ENTITY -> MOLECULE PROFILES")

    links = entity_molecules[["entity_id", "pubchem_id"]].copy()

    links["entity_id"] = pd.to_numeric(links["entity_id"], errors="coerce")
    links["pubchem_id"] = pd.to_numeric(links["pubchem_id"], errors="coerce")
    links = links.dropna(subset=["entity_id", "pubchem_id"])

    links["entity_id"] = links["entity_id"].astype(int)
    links["pubchem_id"] = links["pubchem_id"].astype(int)
    links = links.drop_duplicates(["entity_id", "pubchem_id"])

    molecule_sets = (
        links.groupby("entity_id")["pubchem_id"]
        .apply(set)
        .to_dict()
    )

    alias_columns = [
        "entity_alias_readable",
        "entity_alias",
        "entity_alias_basket",
        "entity_alias_synonyms",
        "natural_source_name",
    ]

    available_alias_columns = [
        c for c in alias_columns if c in entities.columns
    ]

    entity_aliases = {}
    entity_meta = {}

    for _, row in entities.iterrows():
        entity_id = int(row["entity_id"])

        aliases = set()
        for column in available_alias_columns:
            value = row.get(column, "")
            if pd.isna(value):
                continue

            text = str(value).strip()
            if not text:
                continue

            aliases.add(text)

            for part in re.split(r"[|;,/]", text):
                part = part.strip()
                if part:
                    aliases.add(part)

        normalized_aliases = {
            normalize_name(x) for x in aliases if normalize_name(x)
        }

        entity_aliases[entity_id] = normalized_aliases

        meta = {}
        for column in [
            "category",
            "category_readable",
            "natural_source_name",
            "entity_alias_readable",
            "entity_alias",
            "entity_alias_basket",
            "entity_alias_synonyms",
        ]:
            if column in entities.columns:
                meta[column] = clean_text(row.get(column, ""))
            else:
                meta[column] = ""

        entity_meta[entity_id] = meta

    print(f"Entities with molecule profiles : {len(molecule_sets):,}")
    print(f"Entities without profiles       : {len(entities) - len(molecule_sets):,}")
    print(f"Alias columns used              : {', '.join(available_alias_columns)}")

    return molecule_sets, entity_aliases, entity_meta


# ================================================================
# 7. ENTITY PROFILE STATISTICS
# ================================================================

def build_entity_statistics(entities, molecule_sets, entity_aliases):
    section("3. PROFILE SIZE AUDIT")

    rows = []

    for _, row in entities.iterrows():
        entity_id = int(row["entity_id"])
        molecule_set = molecule_sets.get(entity_id, set())
        aliases = entity_aliases.get(entity_id, set())

        rows.append({
            "entity_id": entity_id,
            "entity_name": first_alias(aliases),
            "alias_count": len(aliases),
            "molecule_count": len(molecule_set),
            "small_profile_flag": len(molecule_set) < MIN_PROFILE_SIZE,
            "profile_signature": profile_signature(molecule_set),
        })

    stats = pd.DataFrame(rows)
    positive = stats.loc[stats["molecule_count"] > 0, "molecule_count"]

    print(f"Entities total                  : {len(stats):,}")
    print(f"Entities with molecules         : {len(positive):,}")
    print(f"Entities without molecules      : {(stats['molecule_count'] == 0).sum():,}")
    print(
        f"Profiles with < {MIN_PROFILE_SIZE} molecules : "
        f"{stats['small_profile_flag'].sum():,}"
    )

    if len(positive):
        print(f"Mean molecules/entity            : {positive.mean():.2f}")
        print(f"Median molecules/entity          : {positive.median():.2f}")
        print(f"Min molecules/entity             : {positive.min():.0f}")
        print(f"Max molecules/entity             : {positive.max():.0f}")
        print(f"Q25                              : {positive.quantile(.25):.2f}")
        print(f"Q75                              : {positive.quantile(.75):.2f}")

    return stats


# ================================================================
# 8. MOLECULE FREQUENCY + IDF
# ================================================================

def build_frequency_table(molecule_sets, total_entities):
    section("4. MOLECULE FREQUENCY + IDF")

    counter = Counter()

    for molecule_set in molecule_sets.values():
        for molecule_id in molecule_set:
            counter[molecule_id] += 1

    rows = []
    for molecule_id, frequency in counter.items():
        rows.append({
            "pubchem_id": molecule_id,
            "entity_frequency": frequency,
            "entity_frequency_fraction": frequency / total_entities,
            "idf": smoothed_idf(total_entities, frequency),
        })

    frequency_df = pd.DataFrame(rows)

    if frequency_df.empty:
        return frequency_df, {}

    frequency_df = frequency_df.sort_values(
        ["entity_frequency", "pubchem_id"],
        ascending=[False, True],
    ).reset_index(drop=True)

    weights = dict(
        zip(frequency_df["pubchem_id"], frequency_df["idf"])
    )

    print(f"Unique linked molecules           : {len(frequency_df):,}")
    print(
        f"Most frequent molecule frequency : "
        f"{frequency_df['entity_frequency'].max():,}"
    )
    print(f"Mean IDF                          : {frequency_df['idf'].mean():.4f}")
    print(f"Median IDF                        : {frequency_df['idf'].median():.4f}")
    print(f"Minimum IDF                       : {frequency_df['idf'].min():.4f}")
    print(f"Maximum IDF                       : {frequency_df['idf'].max():.4f}")

    print("\nTop 10 most frequent molecules:")
    print(frequency_df.head(10).to_string(index=False))

    return frequency_df, weights


# ================================================================
# 9. NAME INDEX
# ================================================================

def build_name_index(entity_aliases):
    index = defaultdict(set)
    for entity_id, aliases in entity_aliases.items():
        for alias in aliases:
            index[alias].add(entity_id)
    return index


def resolve_name(name, name_index):
    normalized = normalize_name(name)
    matches = sorted(name_index.get(normalized, set()))

    if len(matches) == 1:
        return matches[0], "EXACT_ALIAS"
    if len(matches) > 1:
        return None, "AMBIGUOUS_ALIAS"
    return None, "NOT_FOUND"


# ================================================================
# 10. PAIR METRICS
# ================================================================

def calculate_pair_metrics(
    entity_a,
    entity_b,
    set_a,
    set_b,
    weights,
):
    shared = set_a & set_b
    union = set_a | set_b

    weighted_shared = weighted_sum(shared, weights)
    weighted_union = weighted_sum(union, weights)
    weighted_a = weighted_sum(set_a, weights)
    weighted_b = weighted_sum(set_b, weights)

    return {
        "entity_id_a": entity_a,
        "entity_id_b": entity_b,
        "molecules_a": len(set_a),
        "molecules_b": len(set_b),
        "profile_size_ratio": (
            min(len(set_a), len(set_b)) / max(len(set_a), len(set_b))
            if set_a and set_b else 0.0
        ),
        "shared_molecules": len(shared),
        "union_molecules": len(union),
        "jaccard": jaccard(set_a, set_b),
        "overlap_coefficient": overlap_coefficient(set_a, set_b),
        "coverage_a": len(shared) / len(set_a) if set_a else 0.0,
        "coverage_b": len(shared) / len(set_b) if set_b else 0.0,
        "symmetric_coverage": symmetric_coverage(set_a, set_b),
        "weighted_shared_idf": weighted_shared,
        "weighted_jaccard_idf": (
            weighted_shared / weighted_union if weighted_union else 0.0
        ),
        "weighted_coverage_a_idf": (
            weighted_shared / weighted_a if weighted_a else 0.0
        ),
        "weighted_coverage_b_idf": (
            weighted_shared / weighted_b if weighted_b else 0.0
        ),
        "low_evidence_flag": (
            len(set_a) < MIN_PROFILE_SIZE
            or len(set_b) < MIN_PROFILE_SIZE
        ),
    }


# ================================================================
# 11. EXPLICIT SANITY PAIRS
# ================================================================

def audit_example_pairs(molecule_sets, entity_aliases, weights):
    section("5. EXPLICIT SANITY-PAIR AUDIT")

    name_index = build_name_index(entity_aliases)
    rows = []

    for requested_a, requested_b in EXAMPLE_PAIRS:
        id_a, status_a = resolve_name(requested_a, name_index)
        id_b, status_b = resolve_name(requested_b, name_index)

        if id_a is None or id_b is None:
            rows.append({
                "ingredient_a": requested_a,
                "ingredient_b": requested_b,
                "status": (
                    "AMBIGUOUS"
                    if "AMBIGUOUS_ALIAS" in (status_a, status_b)
                    else "NOT_FOUND"
                ),
                "match_status_a": status_a,
                "match_status_b": status_b,
            })
            print(f"{requested_a} <-> {requested_b}: {rows[-1]['status']}")
            continue

        row = calculate_pair_metrics(
            id_a,
            id_b,
            molecule_sets.get(id_a, set()),
            molecule_sets.get(id_b, set()),
            weights,
        )

        row.update({
            "ingredient_a": requested_a,
            "ingredient_b": requested_b,
            "status": "OK",
            "match_status_a": status_a,
            "match_status_b": status_b,
        })
        rows.append(row)

        print(
            f"{requested_a} <-> {requested_b}: "
            f"weighted_J={row['weighted_jaccard_idf']:.4f}, "
            f"raw_J={row['jaccard']:.4f}, "
            f"shared={row['shared_molecules']}"
        )

    return pd.DataFrame(rows)


# ================================================================
# 12. ALL-PAIR MOLECULAR SIMILARITY
# ================================================================

def build_pairwise_similarity(molecule_sets, entity_aliases, weights):
    section("6. FINAL MOLECULAR SIMILARITY TABLE")

    molecule_to_entities = defaultdict(list)

    for entity_id, molecule_set in molecule_sets.items():
        for molecule_id in molecule_set:
            molecule_to_entities[molecule_id].append(entity_id)

    # Count candidate pairs by shared molecule.
    shared_counts = Counter()

    for entity_ids in molecule_to_entities.values():
        entity_ids = sorted(set(entity_ids))

        for i, entity_a in enumerate(entity_ids):
            for entity_b in entity_ids[i + 1:]:
                shared_counts[(entity_a, entity_b)] += 1

    profile_sizes = {
        entity_id: len(profile)
        for entity_id, profile in molecule_sets.items()
    }

    weighted_sizes = {
        entity_id: weighted_sum(profile, weights)
        for entity_id, profile in molecule_sets.items()
    }

    names = {
        entity_id: first_alias(entity_aliases.get(entity_id, set()))
        for entity_id in molecule_sets
    }

    rows = []

    for (entity_a, entity_b), shared_count in shared_counts.items():
        set_a = molecule_sets[entity_a]
        set_b = molecule_sets[entity_b]

        shared = set_a & set_b
        size_a = profile_sizes[entity_a]
        size_b = profile_sizes[entity_b]

        weighted_shared = weighted_sum(shared, weights)
        weighted_union = (
            weighted_sizes[entity_a]
            + weighted_sizes[entity_b]
            - weighted_shared
        )

        union_count = size_a + size_b - shared_count

        # PRIMARY MOLECULAR SIMILARITY:
        # IDF-weighted Jaccard.
        # Rare molecules contribute more than ubiquitous molecules.
        molecular_similarity = (
            weighted_shared / weighted_union
            if weighted_union else 0.0
        )

        rows.append({
            "entity_id_a": entity_a,
            "entity_id_b": entity_b,
            "entity_a": names[entity_a],
            "entity_b": names[entity_b],
            "molecules_a": size_a,
            "molecules_b": size_b,
            "profile_size_ratio": (
                min(size_a, size_b) / max(size_a, size_b)
                if max(size_a, size_b) else 0.0
            ),
            "shared_molecules": shared_count,
            "union_molecules": union_count,
            "jaccard": (
                shared_count / union_count if union_count else 0.0
            ),
            "overlap_coefficient": (
                shared_count / min(size_a, size_b)
                if min(size_a, size_b) else 0.0
            ),
            "coverage_a": shared_count / size_a if size_a else 0.0,
            "coverage_b": shared_count / size_b if size_b else 0.0,
            "symmetric_coverage": (
                0.5 * (shared_count / size_a + shared_count / size_b)
                if size_a and size_b else 0.0
            ),
            "weighted_shared_idf": weighted_shared,
            "weighted_jaccard_idf": molecular_similarity,
            "weighted_coverage_a_idf": (
                weighted_shared / weighted_sizes[entity_a]
                if weighted_sizes[entity_a] else 0.0
            ),
            "weighted_coverage_b_idf": (
                weighted_shared / weighted_sizes[entity_b]
                if weighted_sizes[entity_b] else 0.0
            ),
            "low_evidence_flag": (
                size_a < MIN_PROFILE_SIZE
                or size_b < MIN_PROFILE_SIZE
            ),
            "identical_profile_flag": set_a == set_b,
        })

    result = pd.DataFrame(rows)

    if result.empty:
        print("No pairs share molecules.")
        return result

    result = result.sort_values(
        [
            "weighted_jaccard_idf",
            "jaccard",
            "shared_molecules",
        ],
        ascending=False,
    ).reset_index(drop=True)

    print(f"Pairs with >= 1 shared molecule : {len(result):,}")
    print(f"Low-evidence pairs              : {result['low_evidence_flag'].sum():,}")
    print(f"Identical-profile pairs         : {result['identical_profile_flag'].sum():,}")

    print("\nTop 20 molecular-similarity pairs:")
    print(
        result[
            [
                "entity_a",
                "entity_b",
                "molecules_a",
                "molecules_b",
                "shared_molecules",
                "jaccard",
                "weighted_jaccard_idf",
                "low_evidence_flag",
                "identical_profile_flag",
            ]
        ].head(20).to_string(index=False)
    )

    return result


# ================================================================
# 13. IDENTICAL PROFILE INVESTIGATION
# ================================================================

def investigate_identical_profiles(
    entities,
    molecules,
    molecule_sets,
    entity_aliases,
    entity_meta,
):
    section("7. IDENTICAL PROFILE INVESTIGATION")

    profile_groups = defaultdict(list)

    for entity_id, molecule_set in molecule_sets.items():
        profile_groups[tuple(sorted(molecule_set))].append(entity_id)

    duplicate_groups = [
        (profile, ids)
        for profile, ids in profile_groups.items()
        if len(ids) > 1
    ]

    duplicate_groups.sort(
        key=lambda x: (len(x[1]), len(x[0])),
        reverse=True,
    )

    group_rows = []
    entity_rows = []
    molecule_rows = []

    # Molecule metadata lookup.
    molecule_table = molecules.copy()
    molecule_table["pubchem_id"] = pd.to_numeric(
        molecule_table["pubchem_id"],
        errors="coerce",
    )

    molecule_lookup = {
        int(row["pubchem_id"]): row
        for _, row in molecule_table.dropna(subset=["pubchem_id"]).iterrows()
    }

    for group_id, (profile, entity_ids) in enumerate(duplicate_groups, start=1):
        signature = profile_signature(set(profile))
        names = [
            first_alias(entity_aliases.get(entity_id, set()))
            for entity_id in entity_ids
        ]

        categories = {
            entity_meta.get(entity_id, {}).get("category", "")
            for entity_id in entity_ids
        }
        category_readable = {
            entity_meta.get(entity_id, {}).get("category_readable", "")
            for entity_id in entity_ids
        }
        natural_sources = {
            entity_meta.get(entity_id, {}).get("natural_source_name", "")
            for entity_id in entity_ids
        }

        categories.discard("")
        category_readable.discard("")
        natural_sources.discard("")

        group_rows.append({
            "profile_group_id": group_id,
            "profile_signature": signature,
            "entity_count": len(entity_ids),
            "molecule_count": len(profile),
            "entity_ids": safe_join(sorted(entity_ids)),
            "entities": safe_join(sorted(names)),
            "category_count": len(categories),
            "categories": safe_join(sorted(categories)),
            "category_readable_count": len(category_readable),
            "category_readable": safe_join(sorted(category_readable)),
            "natural_source_count": len(natural_sources),
            "natural_sources": safe_join(sorted(natural_sources)),
            "molecule_ids": safe_join(profile),
        })

        for entity_id in sorted(entity_ids):
            meta = entity_meta.get(entity_id, {})
            entity_rows.append({
                "profile_group_id": group_id,
                "profile_signature": signature,
                "entity_id": entity_id,
                "entity_name": first_alias(entity_aliases.get(entity_id, set())),
                "molecule_count": len(profile),
                "category": meta.get("category", ""),
                "category_readable": meta.get("category_readable", ""),
                "natural_source_name": meta.get("natural_source_name", ""),
                "entity_alias_readable": meta.get("entity_alias_readable", ""),
                "entity_alias": meta.get("entity_alias", ""),
                "entity_alias_basket": meta.get("entity_alias_basket", ""),
                "entity_alias_synonyms": meta.get("entity_alias_synonyms", ""),
            })

        for molecule_id in profile:
            row = molecule_lookup.get(molecule_id, {})

            def molecule_value(column):
                value = row.get(column, "")
                return clean_text(value)

            molecule_rows.append({
                "profile_group_id": group_id,
                "profile_signature": signature,
                "molecule_count_in_profile": len(profile),
                "pubchem_id": molecule_id,
                "common_name": molecule_value("common_name"),
                "iupac_name": molecule_value("iupac_name"),
                "odor": molecule_value("odor"),
                "taste": molecule_value("taste"),
                "functional_groups": molecule_value("functional_groups"),
                "flavor_profile": molecule_value("flavor_profile"),
                "fema_flavor_profile": molecule_value("fema_flavor_profile"),
                "fooddb_flavor_profile": molecule_value("fooddb_flavor_profile"),
                "molecular_weight": molecule_value("molecular_weight"),
                "xlogp": molecule_value("xlogp"),
                "heavy_atom_count": molecule_value("heavy_atom_count"),
            })

    groups_df = pd.DataFrame(group_rows)
    entity_df = pd.DataFrame(entity_rows)
    molecule_df = pd.DataFrame(molecule_rows)

    total_duplicate_entities = sum(len(ids) for _, ids in duplicate_groups)

    print(f"Unique molecular profiles       : {len(profile_groups):,}")
    print(f"Identical-profile groups        : {len(duplicate_groups):,}")
    print(f"Entities in duplicate groups   : {total_duplicate_entities:,}")

    if duplicate_groups:
        print("\nLargest identical-profile groups:")
        for group_id, (profile, entity_ids) in enumerate(
            duplicate_groups[:10], start=1
        ):
            names = sorted(
                first_alias(entity_aliases.get(i, set()))
                for i in entity_ids
            )
            print(
                f"Group {group_id}: {len(entity_ids)} entities, "
                f"{len(profile)} molecules"
            )
            print("  " + ", ".join(names[:20]))
            if len(names) > 20:
                print(f"  ... and {len(names) - 20} more")

    return groups_df, entity_df, molecule_df


# ================================================================
# 14. UNRESOLVED NAME CANDIDATES
# ================================================================

def build_candidate_index(entity_aliases, entity_meta, molecule_sets):
    rows = []

    for entity_id, aliases in entity_aliases.items():
        for alias in aliases:
            rows.append({
                "entity_id": entity_id,
                "alias": alias,
                "category": entity_meta.get(entity_id, {}).get("category", ""),
                "category_readable": entity_meta.get(entity_id, {}).get(
                    "category_readable", ""
                ),
                "natural_source_name": entity_meta.get(entity_id, {}).get(
                    "natural_source_name", ""
                ),
                "molecule_count": len(molecule_sets.get(entity_id, set())),
            })

    return rows


def investigate_name_matching(
    molecule_sets,
    entity_aliases,
    entity_meta,
):
    section("8. NAME MATCHING + CANDIDATE GENERATION")

    name_index = build_name_index(entity_aliases)
    rows = []
    candidate_rows = []

    alias_rows = build_candidate_index(
        entity_aliases,
        entity_meta,
        molecule_sets,
    )

    for requested_a, requested_b in EXAMPLE_PAIRS:
        for side, requested_name in [
            ("a", requested_a),
            ("b", requested_b),
        ]:
            normalized = normalize_name(requested_name)
            exact_matches = sorted(name_index.get(normalized, set()))

            if len(exact_matches) == 1:
                entity_id = exact_matches[0]
                aliases = sorted(entity_aliases.get(entity_id, set()))

                rows.append({
                    "pair": f"{requested_a} <-> {requested_b}",
                    "side": side,
                    "requested_name": requested_name,
                    "normalized_name": normalized,
                    "match_status": "EXACT_ALIAS",
                    "matched_entity_id": entity_id,
                    "matched_entity_name": first_alias(aliases),
                    "molecule_count": len(molecule_sets.get(entity_id, set())),
                    "available_aliases": safe_join(aliases),
                })
                continue

            if len(exact_matches) > 1:
                rows.append({
                    "pair": f"{requested_a} <-> {requested_b}",
                    "side": side,
                    "requested_name": requested_name,
                    "normalized_name": normalized,
                    "match_status": "AMBIGUOUS_ALIAS",
                    "matched_entity_id": "",
                    "matched_entity_name": "",
                    "molecule_count": "",
                    "available_aliases": "",
                })
                continue

            # Unresolved: generate candidates, but DO NOT accept one.
            scored = []

            for item in alias_rows:
                alias = item["alias"]
                score = similarity_score(requested_name, alias)

                if score >= MIN_CANDIDATE_SCORE:
                    scored.append((score, item))

            scored.sort(
                key=lambda x: (x[0], x[1]["molecule_count"]),
                reverse=True,
            )

            seen = set()
            top = []

            for score, item in scored:
                key = item["entity_id"]
                if key in seen:
                    continue
                seen.add(key)
                top.append((score, item))
                if len(top) >= TOP_NAME_CANDIDATES:
                    break

            rows.append({
                "pair": f"{requested_a} <-> {requested_b}",
                "side": side,
                "requested_name": requested_name,
                "normalized_name": normalized,
                "match_status": "NOT_FOUND",
                "matched_entity_id": "",
                "matched_entity_name": "",
                "molecule_count": "",
                "available_aliases": "",
            })

            for rank, (score, item) in enumerate(top, start=1):
                candidate_rows.append({
                    "pair": f"{requested_a} <-> {requested_b}",
                    "side": side,
                    "requested_name": requested_name,
                    "candidate_rank": rank,
                    "candidate_entity_id": item["entity_id"],
                    "candidate_alias": item["alias"],
                    "candidate_score": round(score, 6),
                    "candidate_category": item["category"],
                    "candidate_category_readable": item["category_readable"],
                    "candidate_natural_source_name": item["natural_source_name"],
                    "candidate_molecule_count": item["molecule_count"],
                    "mapping_decision": "REVIEW_ONLY",
                })

    matching_df = pd.DataFrame(rows)
    candidates_df = pd.DataFrame(candidate_rows)

    print("\nName matching:")
    for _, row in matching_df.iterrows():
        print(
            f"  {row['requested_name']:<25} "
            f"{row['match_status']}"
            + (
                f" -> {row['matched_entity_name']}"
                if row["match_status"] == "EXACT_ALIAS"
                else ""
            )
        )

    if not candidates_df.empty:
        print("\nTop unresolved-name candidates:")
        print(
            candidates_df[
                [
                    "requested_name",
                    "candidate_alias",
                    "candidate_score",
                    "candidate_category_readable",
                    "candidate_molecule_count",
                ]
            ].head(20).to_string(index=False)
        )

    return matching_df, candidates_df


# ================================================================
# 15. SUMMARY
# ================================================================

def write_summary(
    summary,
    entities,
    molecule_sets,
    frequency_df,
    pairwise_df,
    identical_profiles_df,
    name_matching_df,
    candidates_df,
    missing_ids,
):
    small_profile_count = sum(
        len(profile) < MIN_PROFILE_SIZE
        for profile in molecule_sets.values()
    )

    summary.extend([
        "FINAL MOLECULAR-SIMILARITY DECISION",
        "-" * 72,
        "Primary molecular similarity = IDF-weighted Jaccard.",
        "Formula = sum(IDF of shared molecules) / "
        "sum(IDF of union molecules).",
        "Common molecules contribute less than rare molecules.",
        "Low-evidence profiles are retained but flagged; they should not "
        "be treated as equally reliable downstream.",
        "Identical molecular profiles are explicitly flagged.",
        "Name candidates are review-only and are never silently mapped.",
        "",
        "FINAL AUDIT COUNTS",
        "-" * 72,
        f"FlavorDB entities: {len(entities):,}",
        f"Entities with molecule profiles: {len(molecule_sets):,}",
        f"Small profiles (< {MIN_PROFILE_SIZE}): {small_profile_count:,}",
        f"Unique linked molecules: {len(frequency_df):,}",
        f"Pairs with >= 1 shared molecule: {len(pairwise_df):,}",
        f"Identical-profile groups: {len(identical_profiles_df):,}",
        f"Name checks: {len(name_matching_df):,}",
        f"Generated review candidates: {len(candidates_df):,}",
        f"Missing referenced molecule IDs: {len(missing_ids):,}",
        "",
        "IMPORTANT INTERPRETATION",
        "-" * 72,
        "A high molecular similarity means the two FlavorDB entities "
        "share a large weighted molecular profile.",
        "It does NOT by itself mean that one ingredient is a valid "
        "substitute for the other.",
        "The final substitution framework should combine this feature "
        "with graph/contextual compatibility, carbon footprint, "
        "nutrition preservation, and hard constraints.",
        "",
        "OUTPUTS",
        "-" * 72,
        str(ENTITY_STATS_FILE),
        str(MOLECULE_FREQ_FILE),
        str(PAIR_EXAMPLES_FILE),
        str(PAIRWISE_FILE),
        str(IDENTICAL_PROFILES_FILE),
        str(IDENTICAL_ENTITIES_FILE),
        str(IDENTICAL_MOLECULES_FILE),
        str(NAME_MATCH_FILE),
        str(NAME_CANDIDATES_FILE),
        str(SUMMARY_FILE),
    ])

    SUMMARY_FILE.write_text("\n".join(summary), encoding="utf-8")


# ================================================================
# 16. MAIN
# ================================================================

def main():
    summary = []

    section("FINAL FLAVORDB MOLECULE AUDIT")

    print(
        "Primary molecular similarity: IDF-weighted Jaccard."
    )
    print(
        "This script prepares the molecular feature for the "
        "substitution framework; it does not create the final "
        "multi-objective substitution score."
    )

    entities, entity_molecules, molecules = load_data()

    missing_ids = audit_basic_structure(
        entities,
        entity_molecules,
        molecules,
        summary,
    )

    molecule_sets, entity_aliases, entity_meta = build_profiles(
        entities,
        entity_molecules,
    )

    entity_stats = build_entity_statistics(
        entities,
        molecule_sets,
        entity_aliases,
    )

    frequency_df, weights = build_frequency_table(
        molecule_sets,
        len(entities),
    )

    pair_examples_df = audit_example_pairs(
        molecule_sets,
        entity_aliases,
        weights,
    )

    pairwise_df = build_pairwise_similarity(
        molecule_sets,
        entity_aliases,
        weights,
    )

    identical_profiles_df, identical_entities_df, identical_molecules_df = (
        investigate_identical_profiles(
            entities,
            molecules,
            molecule_sets,
            entity_aliases,
            entity_meta,
        )
    )

    name_matching_df, name_candidates_df = investigate_name_matching(
        molecule_sets,
        entity_aliases,
        entity_meta,
    )

    # ------------------------------------------------------------
    # Save all outputs
    # ------------------------------------------------------------

    entity_stats.to_csv(ENTITY_STATS_FILE, index=False)
    frequency_df.to_csv(MOLECULE_FREQ_FILE, index=False)
    pair_examples_df.to_csv(PAIR_EXAMPLES_FILE, index=False)
    pairwise_df.to_csv(PAIRWISE_FILE, index=False)

    identical_profiles_df.to_csv(
        IDENTICAL_PROFILES_FILE,
        index=False,
    )
    identical_entities_df.to_csv(
        IDENTICAL_ENTITIES_FILE,
        index=False,
    )
    identical_molecules_df.to_csv(
        IDENTICAL_MOLECULES_FILE,
        index=False,
    )

    name_matching_df.to_csv(
        NAME_MATCH_FILE,
        index=False,
    )
    name_candidates_df.to_csv(
        NAME_CANDIDATES_FILE,
        index=False,
    )

    write_summary(
        summary,
        entities,
        molecule_sets,
        frequency_df,
        pairwise_df,
        identical_profiles_df,
        name_matching_df,
        name_candidates_df,
        missing_ids,
    )

    section("AUDIT COMPLETE")

    print(f"Entity statistics      : {ENTITY_STATS_FILE}")
    print(f"Molecule frequency     : {MOLECULE_FREQ_FILE}")
    print(f"Pair examples          : {PAIR_EXAMPLES_FILE}")
    print(f"Pairwise similarity    : {PAIRWISE_FILE}")
    print(f"Identical profiles     : {IDENTICAL_PROFILES_FILE}")
    print(f"Identical entities     : {IDENTICAL_ENTITIES_FILE}")
    print(f"Identical molecules    : {IDENTICAL_MOLECULES_FILE}")
    print(f"Name matching          : {NAME_MATCH_FILE}")
    print(f"Name candidates        : {NAME_CANDIDATES_FILE}")
    print(f"Summary                : {SUMMARY_FILE}")



main()



FINAL FLAVORDB MOLECULE AUDIT
Primary molecular similarity: IDF-weighted Jaccard.
This script prepares the molecular feature for the substitution framework; it does not create the final multi-objective substitution score.

LOADING FLAVORDB DATA
FlavorDB entities     : 936
Entity-molecule links : 60,263
FlavorDB molecules    : 1,788

1. BASIC DATA INTEGRITY
Duplicate entity_id rows          : 0
Duplicate entity-molecule links   : 0
Duplicate molecule-table IDs      : 0
Referenced molecule IDs           : 1,788
Molecule-table IDs                : 1,788
Referenced IDs missing in table   : 0
Unused molecule-table IDs         : 0

2. BUILDING ENTITY -> MOLECULE PROFILES
Entities with molecule profiles : 936
Entities without profiles       : 0
Alias columns used              : entity_alias_readable, entity_alias, entity_alias_synonyms

3. PROFILE SIZE AUDIT
Entities total                  : 936
Entities with molecules         : 936
Entities without molecules      : 0
Profiles with < 3 molec

## 4. Canonical molecular similarity feature

This section retains the validation and feature construction from `02_molecular_similarity.py`. The canonical downstream column is `molecular_similarity`, equal to `weighted_jaccard_idf`.

The audit table contains pairs with at least one shared molecule. Pairs with no shared molecule are not represented in this sparse pair table; absence of a row must not automatically be interpreted as a verified zero-similarity pair.


In [4]:
"""
02_molecular_similarity.py

Purpose
-------
Convert the audited FlavorDB pairwise molecular-overlap results into
the molecular similarity component used by the later substitution
pipeline.

Primary molecular similarity:
    IDF-weighted Jaccard similarity

Important:
    Molecular similarity is evidence of molecular-profile similarity.
    It is NOT, by itself, a substitution suitability score.

The later substitution pipeline will combine this feature with:
    - GraphSAGE / graph-context compatibility
    - carbon footprint
    - nutrition preservation
    - hard constraints

Input
-----
data/molecule_audit_pairwise_overlap.csv

Output
------
data/molecular_similarity.csv
data/molecular_similarity_summary.txt
"""


from pathlib import Path

import numpy as np
import pandas as pd


# ================================================================
# PATHS
# ================================================================


INPUT_FILE = OUTPUT_DIR / "molecule_audit_pairwise_overlap.csv"
OUTPUT_FILE = OUTPUT_DIR / "molecular_similarity.csv"
SUMMARY_FILE = OUTPUT_DIR / "molecular_similarity_summary.txt"


# ================================================================
# REQUIRED COLUMNS
# ================================================================

REQUIRED_COLUMNS = [
    "entity_id_a",
    "entity_id_b",
    "entity_a",
    "entity_b",
    "molecules_a",
    "molecules_b",
    "shared_molecules",
    "jaccard",
    "weighted_jaccard_idf",
    "low_evidence_flag",
    "identical_profile_flag",
]


# ================================================================
# HELPER FUNCTIONS
# ================================================================

def section(title):
    """Print a formatted section heading."""

    print()
    print("=" * 70)
    print(title)
    print("=" * 70)


def clean_boolean_column(series):
    """
    Convert boolean-like CSV values into actual Python booleans.
    """

    if series.dtype == bool:
        return series

    return (
        series
        .astype(str)
        .str.strip()
        .str.lower()
        .map(
            {
                "true": True,
                "false": False,
                "1": True,
                "0": False,
            }
        )
        .fillna(False)
    )


def validate_input_columns(df):
    """Check that the audit file contains everything required."""

    missing = [
        column
        for column in REQUIRED_COLUMNS
        if column not in df.columns
    ]

    if missing:
        raise ValueError(
            "Input file is missing required columns:\n"
            + "\n".join(f"  - {column}" for column in missing)
        )


def validate_similarity_values(df):
    """
    Validate that the IDF-weighted Jaccard values are legitimate.

    Jaccard similarity must lie in [0, 1].
    """

    values = df["weighted_jaccard_idf"]

    invalid = (
        values.isna()
        | ~np.isfinite(values)
        | (values < 0)
        | (values > 1)
    )

    invalid_count = int(invalid.sum())

    if invalid_count > 0:
        raise ValueError(
            "Found "
            f"{invalid_count:,} invalid IDF-weighted Jaccard values."
        )


def validate_pair_structure(df):
    """
    Check basic structural properties of the pairwise table.
    """

    # No self-pairs should exist.
    self_pairs = (
        df["entity_id_a"]
        == df["entity_id_b"]
    )

    self_pair_count = int(self_pairs.sum())

    if self_pair_count > 0:
        raise ValueError(
            f"Found {self_pair_count:,} self-pairs."
        )

    # A pair should have at least one shared molecule because this
    # audit table contains pairs with >= 1 shared molecule.
    invalid_shared = (
        df["shared_molecules"] < 1
    )

    invalid_shared_count = int(invalid_shared.sum())

    if invalid_shared_count > 0:
        raise ValueError(
            "Found pairs with zero shared molecules: "
            f"{invalid_shared_count:,}"
        )


# ================================================================
# LOAD AUDIT DATA
# ================================================================

def load_pairwise_audit():
    """Load and validate the audited pairwise molecular results."""

    if not INPUT_FILE.exists():
        raise FileNotFoundError(
            f"Input file not found:\n{INPUT_FILE}\n\n"
            "Expected:\n"
            "data/molecule_audit_pairwise_overlap.csv"
        )

    print(f"Loading:\n{INPUT_FILE}")

    df = pd.read_csv(INPUT_FILE)

    print(
        f"Rows loaded : {len(df):,}"
    )

    print(
        f"Columns     : {len(df.columns)}"
    )

    validate_input_columns(df)

    df["low_evidence_flag"] = clean_boolean_column(
        df["low_evidence_flag"]
    )

    df["identical_profile_flag"] = clean_boolean_column(
        df["identical_profile_flag"]
    )

    validate_similarity_values(df)

    validate_pair_structure(df)

    return df


# ================================================================
# BUILD MOLECULAR SIMILARITY TABLE
# ================================================================

def build_similarity_table(df):
    """
    Create the clean molecular-similarity table.

    weighted_jaccard_idf is retained as the original audited
    metric and is also exposed as the canonical
    molecular_similarity feature.
    """

    similarity_df = df[
        [
            "entity_id_a",
            "entity_id_b",
            "entity_a",
            "entity_b",
            "molecules_a",
            "molecules_b",
            "profile_size_ratio",
            "shared_molecules",
            "union_molecules",
            "jaccard",
            "overlap_coefficient",
            "coverage_a",
            "coverage_b",
            "symmetric_coverage",
            "weighted_shared_idf",
            "weighted_jaccard_idf",
            "weighted_coverage_a_idf",
            "weighted_coverage_b_idf",
            "low_evidence_flag",
            "identical_profile_flag",
        ]
    ].copy()

    # Canonical feature name used by the later substitution pipeline.
    similarity_df["molecular_similarity"] = (
        similarity_df["weighted_jaccard_idf"]
    )

    return similarity_df


# ================================================================
# SUMMARY STATISTICS
# ================================================================

def calculate_summary(df):
    """Calculate useful diagnostics without filtering the data."""

    similarity = df["molecular_similarity"]

    summary = {}

    summary["pairs"] = len(df)

    summary["unique_entities"] = len(
        set(df["entity_id_a"]).union(
            set(df["entity_id_b"])
        )
    )

    summary["low_evidence_pairs"] = int(
        df["low_evidence_flag"].sum()
    )

    summary["identical_profile_pairs"] = int(
        df["identical_profile_flag"].sum()
    )

    summary["similarity_min"] = float(
        similarity.min()
    )

    summary["similarity_q25"] = float(
        similarity.quantile(0.25)
    )

    summary["similarity_median"] = float(
        similarity.median()
    )

    summary["similarity_mean"] = float(
        similarity.mean()
    )

    summary["similarity_q75"] = float(
        similarity.quantile(0.75)
    )

    summary["similarity_max"] = float(
        similarity.max()
    )

    summary["similarity_zero"] = int(
        (similarity == 0).sum()
    )

    summary["similarity_one"] = int(
        (similarity == 1).sum()
    )

    return summary


# ================================================================
# HIGH-SIMILARITY DIAGNOSTICS
# ================================================================

def print_high_similarity_examples(df, n=20):
    """
    Print the highest molecular-similarity pairs.

    This is diagnostic only.

    A high molecular similarity score does NOT mean that the
    ingredients are valid substitutions.
    """

    section("TOP MOLECULAR-SIMILARITY PAIRS")

    top = (
        df
        .sort_values(
            [
                "molecular_similarity",
                "shared_molecules",
            ],
            ascending=False,
        )
        .head(n)
    )

    columns = [
        "entity_a",
        "entity_b",
        "molecules_a",
        "molecules_b",
        "shared_molecules",
        "molecular_similarity",
        "low_evidence_flag",
        "identical_profile_flag",
    ]

    print(
        top[columns].to_string(index=False)
    )


# ================================================================
# SUMMARY REPORT
# ================================================================

def write_summary(summary):
    """Write the molecular-similarity summary report."""

    lines = []

    lines.append(
        "FLAVORDB MOLECULAR SIMILARITY"
    )
    lines.append("=" * 70)
    lines.append("")
    lines.append(
        "Primary molecular similarity metric:"
    )
    lines.append(
        "IDF-weighted Jaccard similarity"
    )
    lines.append("")
    lines.append(
        "Formula:"
    )
    lines.append(
        "sum(IDF(shared molecules)) / "
        "sum(IDF(union molecules))"
    )
    lines.append("")
    lines.append(
        "This is a molecular-profile similarity feature."
    )
    lines.append(
        "It is NOT a standalone substitution score."
    )
    lines.append("")
    lines.append(
        "PAIRWISE DATA"
    )
    lines.append("-" * 70)
    lines.append(
        f"Pairs                       : {summary['pairs']:,}"
    )
    lines.append(
        f"Unique entities             : "
        f"{summary['unique_entities']:,}"
    )
    lines.append(
        f"Low-evidence pairs         : "
        f"{summary['low_evidence_pairs']:,}"
    )
    lines.append(
        f"Identical-profile pairs    : "
        f"{summary['identical_profile_pairs']:,}"
    )
    lines.append("")
    lines.append(
        "MOLECULAR SIMILARITY DISTRIBUTION"
    )
    lines.append("-" * 70)
    lines.append(
        f"Minimum                     : "
        f"{summary['similarity_min']:.6f}"
    )
    lines.append(
        f"25th percentile            : "
        f"{summary['similarity_q25']:.6f}"
    )
    lines.append(
        f"Median                      : "
        f"{summary['similarity_median']:.6f}"
    )
    lines.append(
        f"Mean                        : "
        f"{summary['similarity_mean']:.6f}"
    )
    lines.append(
        f"75th percentile            : "
        f"{summary['similarity_q75']:.6f}"
    )
    lines.append(
        f"Maximum                     : "
        f"{summary['similarity_max']:.6f}"
    )
    lines.append(
        f"Pairs with similarity = 0  : "
        f"{summary['similarity_zero']:,}"
    )
    lines.append(
        f"Pairs with similarity = 1  : "
        f"{summary['similarity_one']:,}"
    )
    lines.append("")
    lines.append(
        "INTERPRETATION"
    )
    lines.append("-" * 70)
    lines.append(
        "IDF weighting reduces the influence of molecules "
        "that occur in many FlavorDB entities."
    )
    lines.append(
        "Low-evidence profiles are retained and flagged "
        "rather than silently discarded."
    )
    lines.append(
        "Identical molecular profiles are retained and flagged."
    )
    lines.append(
        "A molecular similarity of 1 does not imply that two "
        "ingredients are valid substitutes."
    )
    lines.append("")
    lines.append(
        "The later substitution model should combine this "
        "feature with graph/context compatibility, carbon "
        "impact, nutrition preservation, and constraints."
    )

    with open(
        SUMMARY_FILE,
        "w",
        encoding="utf-8",
    ) as f:
        f.write("\n".join(lines))


# ================================================================
# MAIN
# ================================================================

def main():

    section("FLAVORDB MOLECULAR SIMILARITY")

    print(
        "Purpose: create the molecular similarity feature "
        "for the substitution pipeline."
    )

    print(
        "Primary metric: IDF-weighted Jaccard"
    )

    print(
        "No final substitution score is created here."
    )

    # ------------------------------------------------------------
    # Load audited pairwise results
    # ------------------------------------------------------------

    section("1. LOAD AUDITED PAIRWISE RESULTS")

    audit_df = load_pairwise_audit()

    print(
        "Input validation: PASSED"
    )

    # ------------------------------------------------------------
    # Build clean similarity table
    # ------------------------------------------------------------

    section("2. BUILD MOLECULAR SIMILARITY TABLE")

    similarity_df = build_similarity_table(
        audit_df
    )

    print(
        f"Similarity rows : "
        f"{len(similarity_df):,}"
    )

    print(
        "Canonical feature: molecular_similarity"
    )

    print(
        "Source metric    : weighted_jaccard_idf"
    )

    # ------------------------------------------------------------
    # Summary
    # ------------------------------------------------------------

    section("3. CALCULATE SUMMARY")

    summary = calculate_summary(
        similarity_df
    )

    print(
        f"Pairs                  : "
        f"{summary['pairs']:,}"
    )

    print(
        f"Unique entities        : "
        f"{summary['unique_entities']:,}"
    )

    print(
        f"Low-evidence pairs     : "
        f"{summary['low_evidence_pairs']:,}"
    )

    print(
        f"Identical-profile pairs: "
        f"{summary['identical_profile_pairs']:,}"
    )

    print(
        f"Median similarity      : "
        f"{summary['similarity_median']:.6f}"
    )

    print(
        f"Mean similarity        : "
        f"{summary['similarity_mean']:.6f}"
    )

    # ------------------------------------------------------------
    # Diagnostics
    # ------------------------------------------------------------

    print_high_similarity_examples(
        similarity_df
    )

    # ------------------------------------------------------------
    # Save
    # ------------------------------------------------------------

    section("4. SAVE OUTPUT")

    similarity_df.to_csv(
        OUTPUT_FILE,
        index=False,
    )

    write_summary(
        summary
    )

    print(
        f"Similarity table:\n{OUTPUT_FILE}"
    )

    print(
        f"Summary report:\n{SUMMARY_FILE}"
    )

    section("DONE")

    print(
        "Molecular similarity component created."
    )

    print(
        "Next step:"
    )

    print(
        "Connect this molecular feature to the "
        "RecipeDB ingredient/candidate mapping."
    )



main()



FLAVORDB MOLECULAR SIMILARITY
Purpose: create the molecular similarity feature for the substitution pipeline.
Primary metric: IDF-weighted Jaccard
No final substitution score is created here.

1. LOAD AUDITED PAIRWISE RESULTS
Loading:
/Users/harshgupta/Coding/BTP_2/EDA/results/flavour/molecule_audit_pairwise_overlap.csv
Rows loaded : 360,743
Columns     : 20
Input validation: PASSED

2. BUILD MOLECULAR SIMILARITY TABLE
Similarity rows : 360,743
Canonical feature: molecular_similarity
Source metric    : weighted_jaccard_idf

3. CALCULATE SUMMARY
Pairs                  : 360,743
Unique entities        : 934
Low-evidence pairs     : 76,492
Identical-profile pairs: 10,873
Median similarity      : 0.033156
Mean similarity        : 0.206589

TOP MOLECULAR-SIMILARITY PAIRS
        entity_a         entity_b  molecules_a  molecules_b  shared_molecules  molecular_similarity  low_evidence_flag  identical_profile_flag
          millet breakfast cereal           95           95                95  

## 5. Output inventory

All generated files are in `EDA/results/flavour/`. The key downstream artifact is:

- `molecular_similarity.csv` — pairwise FlavorDB entity IDs/names, similarity metrics, evidence flags, and the canonical `molecular_similarity` feature.

The substitution pipeline integration is intentionally not performed in this notebook; it will be handled separately after this EDA output has been run and checked.


In [5]:
output_files = sorted(p for p in OUTPUT_DIR.iterdir() if p.is_file())
print(f"Output files generated/present: {len(output_files)}")
for path in output_files:
    print(f"{path.name:55s} {path.stat().st_size:>12,} bytes")


Output files generated/present: 20
heavy_atom_count_boxplot.png                                  19,246 bytes
heavy_atom_count_distribution.png                             27,345 bytes
molecular_similarity.csv                                  81,198,630 bytes
molecular_similarity_summary.txt                               1,505 bytes
molecular_statistics.csv                                         355 bytes
molecular_weight_boxplot.png                                  20,738 bytes
molecular_weight_distribution.png                             27,869 bytes
molecule_audit_entity_stats.csv                               62,405 bytes
molecule_audit_identical_profile_entities.csv                 52,471 bytes
molecule_audit_identical_profile_molecules.csv               119,528 bytes
molecule_audit_identical_profiles.csv                         16,967 bytes
molecule_audit_molecule_frequency.csv                         85,824 bytes
molecule_audit_name_candidates.csv                             2,